# 🐾 Animal Sound Generator — Colab Training

| GPU | VRAM | AE Time | VAE Time |
|-----|------|---------|----------|
| T4 (free) | 16 GB | ~2 hrs | ~2 hrs |
| L4 (pro) | 24 GB | ~1.2 hrs | ~1.2 hrs |

### Before running:
1. Upload `animal_audio.tar.gz` to Google Drive root (`MyDrive/`)
2. Runtime → Change runtime type → **L4 GPU** (or T4 if free)
3. Run cells **top to bottom**

In [ ]:
# @title 1. Setup — Clone & Install

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/grindydev/animal_sound_generator.git /content/animal_sound_generator
%cd /content/animal_sound_generator

!pip install -q torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy matplotlib pandas scikit-learn librosa soundfile tqdm

!mkdir -p models/autoencoder_checkpoints/train
!mkdir -p models/vae_checkpoints/train

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# @title 2. Load Dataset from Drive

import os, tarfile

LOCAL = '/content/animal_sound_generator/data'
TAR = '/content/drive/MyDrive/animal_audio.tar.gz'

if os.path.isdir(os.path.join(LOCAL, 'animal_audio')):
    print('✅ Data already loaded')
elif os.path.exists(TAR):
    print(f'📂 Extracting (this takes ~30s)...')
    os.makedirs(LOCAL, exist_ok=True)
    with tarfile.open(TAR, 'r:gz') as tf:
        tf.extractall(path=LOCAL, filter='data')
    # Fix nested extraction if tar was created with parent folder
    nested = os.path.join(LOCAL, 'data', 'animal_audio')
    if os.path.isdir(nested):
        !mv {nested} {LOCAL}/animal_audio
        !rmdir {LOCAL}/data
    print('✅ Done!')
else:
    print('❌ animal_audio.tar.gz not found! Upload to Drive root.')

!ls data/animal_audio/

In [ ]:
# @title 3. Train Autoencoder (~1-2 hrs)

# Config: lr=1e-3, batch=16, base_ch=32 → 149M params
!python src/vae/train_ae.py

In [ ]:
# @title 4. Train VAE (Fine-tune) (~1-2 hrs)

# Config: lr=3e-4, batch=8, base_ch=32 → 223M params
# Requires best_autoencoder_train.pth from step 3
!python src/vae/finetune.py

In [ ]:
# @title 5. (Optional) Train Classifier (~1 min)

# Only if you don't have best_audio_cnn_train.pth
# Already included in repo at 95% accuracy — usually skip this
!python src/train_classifier.py

In [ ]:
# @title 6. (Optional) Train Diffusion (~3 hrs)

!python src/diffusion/train.py

---
## 💾 Save All Models to Drive

Run this after training — syncs everything to Drive in one shot.

In [ ]:
# @title 💾 Save All to Drive

DRIVE = '/content/drive/MyDrive/animal_sound_generator/models'
!mkdir -p {DRIVE}

models_to_save = [
    'best_autoencoder_train.pth',
    'best_vae_finetune_train.pth',
    'best_audio_cnn_train.pth',
    'diffusion_unet_train_best.pth',
    'hifigan_generator_train.pth',
]

checkpoint_dirs = [
    'autoencoder_checkpoints',
    'vae_checkpoints',
    'diffusion_checkpoints',
    'hifigan_checkpoints',
]

# Save best models
for f in models_to_save:
    path = f'models/{f}'
    if os.path.exists(path):
        !cp {path} {DRIVE}/
        size = os.path.getsize(path) / 1e6
        print(f'  ✅ {f} ({size:.0f} MB)')

# Save checkpoints (for resume)
for d in checkpoint_dirs:
    path = f'models/{d}'
    if os.path.isdir(path):
        !cp -r {path} {DRIVE}/
        print(f'  ✅ {d}/')

print(f'\n📂 All saved to {DRIVE}/')
!ls -lh {DRIVE}/

---
## 🔁 Resume Training (after timeout)

If Colab disconnects, start a new session and run this cell FIRST, then re-run the training cell you were on. Checkpoints auto-resume.

In [ ]:
# @title 🔁 Resume from Drive

DRIVE_MODELS = '/content/drive/MyDrive/animal_sound_generator/models'
LOCAL_MODELS = '/content/animal_sound_generator/models'

import os
if os.path.isdir(DRIVE_MODELS):
    !cp -r {DRIVE_MODELS}/* {LOCAL_MODELS}/ 2>/dev/null
    print('✅ Checkpoints restored from Drive')
    !ls {LOCAL_MODELS}/*.pth 2>/dev/null || echo '(no .pth files yet)'
else:
    print('⚠️  No checkpoints in Drive yet')

# Re-extract data (VM was reset)
import tarfile
TAR = '/content/drive/MyDrive/animal_audio.tar.gz'
LOCAL = '/content/animal_sound_generator/data'
if not os.path.isdir(os.path.join(LOCAL, 'animal_audio')) and os.path.exists(TAR):
    os.makedirs(LOCAL, exist_ok=True)
    with tarfile.open(TAR, 'r:gz') as tf:
        tf.extractall(path=LOCAL, filter='data')
    nested = os.path.join(LOCAL, 'data', 'animal_audio')
    if os.path.isdir(nested):
        !mv {nested} {LOCAL}/animal_audio && rmdir {LOCAL}/data
    print('✅ Data re-extracted')
print('Ready — re-run the training cell you were on')


---
## 🎧 Quick Test — Generate a Sound

After VAE is trained, test the pipeline.

In [ ]:
# @title 🎧 Generate & Listen

!python src/generate.py --label Dog --no-diff --temperature 0.7

from IPython.display import Audio, display
import glob

wavs = sorted(glob.glob('generated_audio/*.wav'))
if wavs:
    display(Audio(wavs[-1], rate=22050))
    print(f'🔊 {wavs[-1]}')
else:
    print('No audio generated — check errors above')